In [1]:
import pandas as pd 
import gzip
import jsonlines
import urllib
from unidecode import unidecode

def parse_path(url):
    path = url.split(')')[-1]
    path = path.split('sfchronicle.com')[-1]
    path = urllib.parse.unquote(path, encoding='utf-8', errors='replace')
    path = unidecode(path).replace(' ', '')
    return path.lower()

import os 

In [5]:
fname = 'cache/full-article-df-for-matching.csv.gz'
if not os.path.exists(fname):
    all_fetched_articles = []
    for fname in [
        '../data/articles/sfchron-cc-fetched-articles.jsonl',
        '../data/articles/sfchron-cc-fetched-articles-2.jsonl.gz',
        '../data/articles/sfchron-fetched-articles.jsonl', 
    ]:
        if fname.endswith('.gz'):
            f = gzip.open(fname)
        else:
            f = open(fname)
        lines = list(jsonlines.Reader(f))
        all_fetched_articles += lines
    
    all_fetched_articles_df = pd.DataFrame(all_fetched_articles)
    all_fetched_articles_df['path'] = all_fetched_articles_df['article_url'].apply(parse_path)
    all_fetched_articles_df = all_fetched_articles_df.drop_duplicates('article_text')
    all_fetched_articles_df['article_publish_date'] = (
        pd.to_datetime(all_fetched_articles_df['article_publish_date'], errors='coerce', utc=True)
    )
    
    (all_fetched_articles_df
         [['article_text', 'article_publish_date', 'path']]
         .to_csv(fname, compression='gzip')
    )
else:
    all_fetched_articles_df = pd.read_csv(fname, index_col=0).reset_index(drop=True)
    all_fetched_articles_df['article_publish_date'] = pd.to_datetime(
        all_fetched_articles_df['article_publish_date']
    )

# Problem Setup

In [8]:
import gdown
import re 

In [9]:
full_meeting_row_df = pd.read_csv('cache/2023-04-13__full_meeting_row_df.csv')
full_meeting_row_df = full_meeting_row_df.assign(text=lambda df: df['Name'] + '. ' + df['Title']).drop(columns=['Name', 'Title'])
full_meeting_row_df['date'] = pd.to_datetime(full_meeting_row_df['date'])

In [10]:
all_candidate_articles = pd.read_json('cache/2023-04-19__candidate-articles-filtered-by-keyword.jsonl', lines=True)

In [12]:
import os 
import pandas as pd 

fname = 'cache/y-true-matched-df.csv.gz'
if not os.path.exists(fname):
    output = 'cache/gold-matching.csv'
    url = 'https://docs.google.com/spreadsheets/d/1T319_gMlJeMcl0T56aeTGjm12havON379fP0W2aJmZA/edit?usp=sharing'
    gdown.download(url, output, quiet=True)

    first_round_annotation_df = (
        pd.read_excel(output)
        .rename(columns=lambda x: re.sub('\s+', ' ', x.replace('\n', ' ')))
        .rename(columns={'text': 'article', 'City Council Meeting Date': 'meeting_date'})
        .assign(method='full_label_first_round')
    )
    
    sfbos_first_round_annotation_df = (
        first_round_annotation_df.loc[lambda df: df['City Council Meeting Minute URL'].str.contains('sfbos') == True]
    )
    sfbos_first_round_annotation_df['meeting_id'] = sfbos_first_round_annotation_df['Page/paragraph'].str.split(' ').str.get(-1)
    sfbos_first_round_annotation_df = sfbos_first_round_annotation_df.loc[lambda df: df['meeting_id'].str.len() == 6]
    sfbos_first_round_annotation_df['meeting_id'] = sfbos_first_round_annotation_df['meeting_id'].astype(int)
    sfbos_first_round_annotation_df = sfbos_first_round_annotation_df.merge(
        full_meeting_row_df[['File #', 'text']].drop_duplicates(), 
        how='left',
        left_on='meeting_id',
        right_on='File #', 
    ).rename(columns={'text': 'meeting_text'})
    
    sfbos_first_round_annotation_df.to_csv(fname, compression='gzip', )
else:
    sfbos_first_round_annotation_df = pd.read_csv(fname)

<>:12: SyntaxWarning: invalid escape sequence '\s'
<>:12: SyntaxWarning: invalid escape sequence '\s'
/var/folders/xh/qnyq7yzj0r328_7hnb7pgxth0000gp/T/ipykernel_96297/1577914749.py:12: SyntaxWarning: invalid escape sequence '\s'
  .rename(columns=lambda x: re.sub('\s+', ' ', x.replace('\n', ' ')))


In [17]:
output = 'cache/full-automated-meeting-article-matching__annotated.xlsx'
end_to_end_annotated_df = pd.read_excel(output)
end_to_end_annotated_df = end_to_end_annotated_df.loc[lambda df: df['Label'].notnull()]
all_candidate_articles = pd.read_json('cache/2023-04-19__candidate-articles-filtered-by-keyword.jsonl', lines=True)

In [18]:
(
    full_meeting_row_df
         .loc[lambda df: df['text'].str.lower().str.contains('sales tax') == True]
         # .loc[lambda df: df['text'].str.lower().str.contains('barred') == True]
         # .iloc[0]['text']
     ['text'].drop_duplicates().tolist()
)

["Initiative Ordinance - Business and Tax Regulations Code - Three-Quarter Cent Sales Tax Increase. Ordinance amending the Business and Tax Regulations Code to impose a transactions (sales) and use tax at the rate of three-quarters of one percent (0.75%) for a period of 25 years, to be administered by the State Board of Equalization in accordance with Parts 1.6 and 1.7 of Division 2 of the California Revenue and Taxation Code; and increasing the City's appropriations limit by the amount of the tax increase for four years from November 8, 2016; and directing submission of the tax for voter approval at the November 8, 2016, general municipal election.",
 'Initiative Ordinance - Business and Tax Regulations Code - Half-Cent Sales Tax Increase for Transportation. Ordinance amending the Business and Tax Regulations Code to impose a transactions (sales) and use tax at the rate of one-half of one percent (0.5%) for 25 years, to be imposed by the San Francisco County Transportation Authority a

In [19]:
(end_to_end_annotated_df
     .loc[lambda df: df['Label'] == 1]
     # .loc[lambda df: df['proposal text'].str.lower().str.contains('bart')][['proposal text', 'article summary']]
     # .iloc[7].to_dict()
    [['proposal text', 'article summary']]
     # .drop_duplicates('article_summary')
     .iloc[10]
 .to_dict()
)

{'proposal text': "Approving Submission of Sales Tax to Support Caltrain Service - November 3, 2020, Election. Resolution approving the Peninsula Corridor Joint Powers Board's placement of a three-county measure to impose a one-eighth of one percent (0.125%) retail transactions and use tax to be used for operating and capital purposes of the Caltrain rail service at an election to be held on November 3, 2020; rescinding and replacing Resolution No. 334-20, which conditionally approved submission of the same measure; and affirming the San Francisco Municipal Transportation Agency's determination under the California Environmental Quality Act.",
 'article summary': 'Their announcement came after the San Francisco Board of Supervisors scuttled an 1/8-cent sales-tax measure that Caltrain had pitched for the November ballot.\n\nThe chief of BART expects the rail system to lose $975 million over three years as sales tax revenues sink and people avoid mass transit -- a scar of COVID-19 that c

# Generate Embeddings (for Comparison)

In [9]:
## openai
from tqdm.auto import tqdm
with jsonlines.open('openai-embeddings.jsonl', 'w') as f:
    for t in tqdm(df['0'], total=len(df)):
        try:
            response = openai.Embedding.create(
                input=t[:25_000],
                model="text-embedding-ada-002"
            )
            embeddings = response['data'][0]['embedding']
        except Exception as e:
            print(f'Exception: {e}')
            embeddings = []
        f.write({'data': embeddings})

NameError: name 'df' is not defined

In [186]:
## openai
from tqdm.auto import tqdm
import openai
openai.api_key_path = '/Users/spangher/.openai-isi-key.txt'
output_name = 'cache/y-true-openai-embeddings.jsonl'
with jsonlines.open(output_name, 'w') as f:
    for t in tqdm(sfbos_first_round_annotation_df['meeting_text'], total=len(sfbos_first_round_annotation_df)):
        try:
            response = openai.Embedding.create(
                input=t[:25_000],
                model="text-embedding-ada-002"
            )
            embeddings = response['data'][0]['embedding']
        except Exception as e:
            print(f'Exception: {e}')
            embeddings = []
        f.write({'data': embeddings})

  0%|          | 0/56 [00:00<?, ?it/s]

In [ ]:
from transformers import AutoModel
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers import models, losses
from torch.utils.data import DataLoader
import pandas as pd

model_name = 'google/bigbird-roberta-large'
word_embedding_model = models.Transformer(model_name, max_seq_length=2000)
pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension())
model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
train_sentences = pd.read_csv('all-text-to-score-sbert.csv.gz')['0'].drop_duplicates()
train_sentences_sample = train_sentences.sample(150_000)
train_data = [InputExample(texts=[s, s]) for s in train_sentences]
train_dataloader = DataLoader(train_data, batch_size=128, shuffle=True)
train_loss = losses.MultipleNegativesRankingLoss(model)
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=10,
    show_progress_bar=True
)

# All articles

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from unidecode import unidecode

In [26]:
tfidf = TfidfVectorizer(min_df=.001, max_df=.5, stop_words='english')
all_vecs = tfidf.fit_transform(
    pd.concat([
        full_meeting_row_df['text'],
        all_fetched_articles_df['article_text']
    ]).fillna('')
)

article_vecs = all_vecs[len(full_meeting_row_df):]
y_true_meeting_vecs = tfidf.transform(sfbos_first_round_annotation_df['meeting_text'])
sims = cosine_similarity(article_vecs, y_true_meeting_vecs)
sim_df = pd.DataFrame(
    sims, 
    columns=sfbos_first_round_annotation_df['meeting_id'], 
    index=all_fetched_articles_df['path']
)

In [27]:
y_true_urls =  (
    sfbos_first_round_annotation_df
         .set_index('meeting_id')['News Article URL']
         .apply(parse_path)
)

In [28]:
(sim_df.idxmax(axis=0) == y_true_urls).mean()

0.17857142857142858

In [29]:
#### embedding comparison

In [30]:
import numpy as np 
sbert_vecs = np.loadtxt('cache/all-MiniLM-L6-v2-sent-embs.txt')
article_sbert_embs = sbert_vecs[len(full_meeting_row_df):]
y_true_meeting_embs = np.loadtxt('cache/y-true-sbert-embs.txt')

In [31]:
sims = cosine_similarity(article_sbert_embs, y_true_meeting_embs)
sim_df = pd.DataFrame(
    sims, 
    columns=sfbos_first_round_annotation_df['meeting_id'], 
    index=all_fetched_articles_df['path']
)

y_true_urls =  (
    sfbos_first_round_annotation_df
         .set_index('meeting_id')['News Article URL']
         .apply(parse_path)
)

In [32]:
(sim_df.idxmax(axis=0) == y_true_urls).mean()

0.32142857142857145

In [33]:
import gzip
import json
import numpy as np 
from tqdm.auto import tqdm

openai_vecs = []
for line in tqdm(gzip.open('cache/openai-embeddings.jsonl.gz', 'rb')):
    vecs = np.array(json.loads(line.decode())['data'])
    openai_vecs.append(vecs)

openai_vecs = list(map(lambda x: x if len(x) == 1536 else np.zeros(1536), openai_vecs))
openai_vecs = np.vstack(openai_vecs)

0it [00:00, ?it/s]

In [34]:
y_true_openai_meeting_embs = (
    pd.read_json('cache/y-true-openai-embeddings.jsonl', lines=True)['data']
        .pipe(lambda df: np.array(df.tolist()))
)

In [35]:
article_openai_embs = openai_vecs[len(full_meeting_row_df):]

In [36]:
sims = cosine_similarity(article_openai_embs, y_true_openai_meeting_embs)
sim_df = pd.DataFrame(
    sims, 
    columns=sfbos_first_round_annotation_df['meeting_id'], 
    index=all_fetched_articles_df['path']
)

y_true_urls =  (
    sfbos_first_round_annotation_df
         .set_index('meeting_id')['News Article URL']
         .apply(parse_path)
)

In [37]:
(sim_df.idxmax(axis=0) == y_true_urls).mean()

0.30357142857142855

# Articles filtered by keyword

In [38]:
candidate_rough_df = (
    all_fetched_articles_df
         .loc[lambda df: 
              df['article_text'].str.contains('Board of Supervisors', case=False) |
              df['article_text'].str.contains('supervisor', case=False) 
         ]
)

In [ ]:
l1_tfidf = TfidfVectorizer(min_df=.001, max_df=.5, stop_words='english')
all_vecs = l1_tfidf.fit_transform(
    pd.concat([
        full_meeting_row_df['text'].sample(10_000),
#         all_candidate_articles['summary_text']
        candidate_rough_df['article_text']
    ]).fillna('')
)

candidate_vecs = all_vecs[10_000:]
y_true_meeting_vecs = l1_tfidf.transform(sfbos_first_round_annotation_df['meeting_text'])
sims = cosine_similarity(candidate_vecs, y_true_meeting_vecs)
sims_1_df = pd.DataFrame(
    sims, 
#     index=all_candidate_articles['article_url'].apply(parse_path), 
    index=candidate_rough_df['article_url'].apply(parse_path), 
    columns=sfbos_first_round_annotation_df['File #']
)

(sims_1_df.idxmax(axis=0) == y_true_urls).mean()

In [40]:
### embedding comparison

In [41]:
sims = cosine_similarity(article_sbert_embs[candidate_rough_df.index], y_true_meeting_embs)
sims_1_df = pd.DataFrame(
    sims, 
    index=candidate_rough_df['path'], 
    columns=sfbos_first_round_annotation_df['File #']
)

(sims_1_df.idxmax(axis=0) == y_true_urls).mean()

0.3392857142857143

In [42]:
sims = cosine_similarity(article_openai_embs[candidate_rough_df.index], y_true_openai_meeting_embs)
sims_1_df = pd.DataFrame(
    sims, 
    index=candidate_rough_df['path'], 
    columns=sfbos_first_round_annotation_df['File #']
)

(sims_1_df.idxmax(axis=0) == y_true_urls).mean()

0.375

# Articles l2

In [43]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.pipeline import Pipeline
from datetime import timedelta

def get_similar_articles(
    meeting_text,
    meeting_date,
    article_df,
    k=4,
    window=30,
    vectorizer_model=None,
    vectorizer_func=lambda model, text_list, text_id: model.transform(text_list),
    meeting_text_idx=None,
):
    query_vec = vectorizer_func(vectorizer_model, [meeting_text], [meeting_text_idx])
    date_limited_article_df = (
        article_df
            .loc[lambda df: df['article_publish_date'].notnull()]
            .loc[lambda df: df['article_publish_date'].dt.date > meeting_date.date() - timedelta(days=window)]
            .loc[lambda df: df['article_publish_date'].dt.date <= meeting_date.date() + timedelta(days=window)]
    )
    if len(date_limited_article_df) == 0:
        return None 

    key_vecs = vectorizer_func(
        vectorizer_model,
        date_limited_article_df['article_text'], 
        date_limited_article_df.index
    )
    sims = cosine_similarity(query_vec, key_vecs).flatten()
    date_limited_article_df['sims'] = sims
    return date_limited_article_df.sort_values('sims', ascending=False).iloc[:k]

In [44]:
output = 'cache/word-matching-annotated-meeting.xlsx'
l2_output_1 = pd.read_excel(output)
output = 'cache/lr-annotated-meeting.xlsx'
l2_output_2 = pd.read_excel(output)
l2_output = pd.concat([l2_output_1, l2_output_2])

In [45]:
pos_examples = l2_output.loc[lambda df: df['Meeting'] == 'SFBOS'][['article', 'Label']]
neg_examples = (
    l2_output
         .loc[lambda df: df['Meeting'] != 'SFBOS']
         .loc[lambda df: df['Label'].notnull()]
         .assign(Label=0)
    [['article', 'Label']]
)
train_df = pd.concat([pos_examples, neg_examples])

In [46]:
l2_pipe = Pipeline([
    ('cv', TfidfVectorizer(min_df=.01, max_df=.5, stop_words='english')),
    ('lr', LogisticRegressionCV())
])

In [47]:
l2_pipe.fit(train_df['article'], y=train_df['Label'])

Pipeline(steps=[('cv',
                 TfidfVectorizer(max_df=0.5, min_df=0.01,
                                 stop_words='english')),
                ('lr', LogisticRegressionCV())])

In [48]:
y_probs = l2_pipe.predict_proba(candidate_rough_df['article_text'])

In [49]:
pd.Series(y_probs.argmax(axis=1)).value_counts()

0    10797
1     2919
Name: count, dtype: int64

In [50]:
candidate_rough_df['l2_prob'] = y_probs[:, 1]

/var/folders/xh/qnyq7yzj0r328_7hnb7pgxth0000gp/T/ipykernel_96297/3269398095.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  candidate_rough_df['l2_prob'] = y_probs[:, 1]


In [51]:
l2_filtered_articles = candidate_rough_df.loc[lambda df: df['l2_prob'] > .1]
l2_tfidf = TfidfVectorizer(min_df=.001, max_df=.5, stop_words='english')
all_vecs = l2_tfidf.fit_transform(
    pd.concat([
        full_meeting_row_df['text'].sample(10_000),
#         l2_filtered_articles['summary_text'],
        l2_filtered_articles['article_text'],
    ]).fillna('')
)

candidate_vecs = all_vecs[10_000:]
y_true_meeting_vecs = l2_tfidf.transform(sfbos_first_round_annotation_df['meeting_text'])
if 'path' not in l2_filtered_articles.columns:
    l2_filtered_articles['path'] = l2_filtered_articles['article_url'].apply(parse_path)

l2_filtered_articles['article_publish_date'] = pd.to_datetime(l2_filtered_articles['article_publish_date'])
sims = cosine_similarity(candidate_vecs, y_true_meeting_vecs)
sims_2_df = pd.DataFrame(
    sims, 
    index=l2_filtered_articles['path'], 
#     index=candidate_rough_df['article_url'].apply(parse_path), 
    columns=sfbos_first_round_annotation_df['File #']
)

(sims_2_df.idxmax(axis=0) == y_true_urls).mean()

/var/folders/xh/qnyq7yzj0r328_7hnb7pgxth0000gp/T/ipykernel_96297/3205761829.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  l2_filtered_articles['article_publish_date'] = pd.to_datetime(l2_filtered_articles['article_publish_date'])


0.32142857142857145

In [55]:
import datetime, dateutil

In [59]:
mapping = {}
for idx, (meeting_id, meeting_text, meeting_date) in (
    sfbos_first_round_annotation_df
        [['meeting_id', 'meeting_text', 'meeting_date']]
        .iterrows()
):
    if isinstance(meeting_date, str):
        meeting_date = dateutil.parser.parse(meeting_date)
    t = get_similar_articles(meeting_text, meeting_date, l2_filtered_articles, k=100, window=30, vectorizer_model=l2_tfidf)
    mapping[meeting_id] = t['path'].tolist()

In [60]:
(
    pd.Series(mapping).str.get(0)
         .to_frame('y_pred')
         .merge(y_true_urls.to_frame('y_true'), right_index=True, left_index=True)
         .pipe(lambda df: df['y_pred'] == df['y_true']).mean()
)

0.5535714285714286

In [61]:
## with embeddings

In [62]:
def get_emb(model, text_list, text_id):
    if len(text_id) == 1:
        return model['meeting_embs'][text_id]
    else:
        return model['article_embs'][text_id]

In [63]:
sfbos_first_round_annotation_df['meeting_date'] = pd.to_datetime(sfbos_first_round_annotation_df['meeting_date'])
mapping = {}
for idx, (meeting_id, meeting_text, meeting_date) in (
    sfbos_first_round_annotation_df
        [['meeting_id', 'meeting_text', 'meeting_date']]
        .iterrows()
):
    t = get_similar_articles(
        meeting_text, 
        meeting_date,
        l2_filtered_articles,
        k=100,
        window=30,
        meeting_text_idx=idx,
        vectorizer_model={'article_embs': article_sbert_embs, 'meeting_embs': y_true_meeting_embs},
        vectorizer_func=get_emb
    )
    mapping[meeting_id] = t['path'].tolist()

In [64]:
(pd.Series(mapping).str.get(0)
 .to_frame('y_pred')
 .merge(y_true_urls.to_frame('y_true'), right_index=True, left_index=True)
 .pipe(lambda df: df['y_pred'] == df['y_true']).mean()
)

0.48214285714285715

In [65]:
sfbos_first_round_annotation_df['meeting_date'] = pd.to_datetime(sfbos_first_round_annotation_df['meeting_date'])
mapping = {}
for idx, (meeting_id, meeting_text, meeting_date) in (
    sfbos_first_round_annotation_df
        [['meeting_id', 'meeting_text', 'meeting_date']]
        .iterrows()
):
    t = get_similar_articles(
        meeting_text, 
        meeting_date,
        l2_filtered_articles,
        k=100,
        window=30,
        meeting_text_idx=idx,
        vectorizer_model={
            'article_embs': article_openai_embs, 
            'meeting_embs': y_true_openai_meeting_embs
        },
        vectorizer_func=get_emb
    )
    mapping[meeting_id] = t['path'].tolist()

In [66]:
(pd.Series(mapping).str.get(0)
 .to_frame('y_pred')
 .merge(y_true_urls.to_frame('y_true'), right_index=True, left_index=True)
 .pipe(lambda df: df['y_pred'] == df['y_true']).mean()
)

0.5357142857142857

# L3

In [69]:
(pd.Series(mapping)#.str.get(0)
 .to_frame('y_pred')
 .merge(y_true_urls.to_frame('y_true'), right_index=True, left_index=True)
 .apply(lambda x: x['y_true'] in x['y_pred'], axis=1)
#  .pipe(lambda df: df['y_pred'] == df['y_true']).mean()
).mean()

0.7321428571428571

# BMP20

In [1]:
import scipy

In [4]:
from retriv import SearchEngine
collection = [
  {"id": "doc_1", "text": "Generals gathered in their masses"},
  {"id": "doc_2", "text": "Just like witches at black masses"},
  {"id": "doc_3", "text": "Evil minds that plot destruction"},
  {"id": "doc_4", "text": "Sorcerer of death's construction"},
]

In [15]:
article_collection = (all_fetched_articles_df
 [['path', 'article_text']]
 .rename(columns={'path': 'id', 'article_text': 'text'})
 .to_dict(orient='rows')
)

/var/folders/xh/qnyq7yzj0r328_7hnb7pgxth0000gp/T/ipykernel_44677/1241168551.py:1: FutureWarning: Using short name for 'orient' is deprecated. Only the options: ('dict', list, 'series', 'split', 'records', 'index') will be used in a future version. Use one of the above to silence this warning.
  article_collection = (all_fetched_articles_df


In [16]:
se = SearchEngine("article-collection").index(article_collection)

Building inverted index: 100%|██████████████████████████████████████████████████| 274498/274498 [00:23<00:00, 11598.53it/s]


In [24]:
top_docs = se.search("witches masses", return_docs=False, cutoff=len(article_collection))

In [ ]:
all_fetched_articles_df.merge(pd.Series(top_docs).to_frame('score'), right_index=True, left_on='path')

In [57]:
from datetime import timedelta
def get_similar_articles_bmp_20(meeting_text, meeting_date, article_df, k=4, window=30, se=None):
    top_docs = se.search(meeting_text, return_docs=False, cutoff=200_000)
    top_article_df = (
        article_df
            .merge(pd.Series(top_docs).to_frame('score'), right_index=True, left_on='path')
    )
    date_limited_article_df = (
        top_article_df
            .loc[lambda df: df['article_publish_date'].notnull()]
            .loc[lambda df: df['article_publish_date'].dt.date > meeting_date.date() - timedelta(days=window)]
            .loc[lambda df: df['article_publish_date'].dt.date <= meeting_date.date() + timedelta(days=window)]
#             .loc[lambda df: df['committee'] == 'Board of Supervisors']
    )
    if len(date_limited_article_df) == 0:
        return None 
    
    return date_limited_article_df.sort_values('score', ascending=False).iloc[:k]

In [59]:
from tqdm.auto import tqdm

In [ ]:
mapping = {}
for idx, (meeting_id, meeting_text, meeting_date) in tqdm(
    sfbos_first_round_annotation_df[['meeting_id', 'meeting_text', 'meeting_date']].iterrows(), total=len(sfbos_first_round_annotation_df)
):
    t = get_similar_articles_bmp_20(
        meeting_text, meeting_date, all_fetched_articles_df, k=4, window=30, se=se)
    mapping[meeting_id] = t['path'].tolist()

In [65]:
(pd.Series(mapping)#.str.get(0)
 .to_frame('y_pred')
 .merge(y_true_urls.to_frame('y_true'), right_index=True, left_index=True)
 .apply(lambda x: x['y_true'] in x['y_pred'], axis=1)
#  .pipe(lambda df: df['y_pred'] == df['y_true']).mean()
).mean()

0.6071428571428571

# Dense Retriever

In [ ]:
from retriv import DenseRetriever

dr = DenseRetriever(
  index_name="dense-index",
  model="sentence-transformers/all-MiniLM-L6-v2",
  normalize=True,
  max_length=128,
  use_ann=True,
)

In [ ]:
SearchEngine(model=)